# Does Housing Supply Constrain Home Prices? Evidence from U.S. Metro Areas

### ECON 320 Lab — Final Project
**Student names:** Tessa Butler, Siya Kumar, Ania Ting, Naman Keswani, Shunji Lewandowski  
**Lab Section:** [Lab #2]  

---
## Abstract
We study whether housing supply constrains home prices across U.S. Metropolitan Statistical Areas (MSAs) using a cross-sectional OLS framework. The unit of observation is an MSA (N ≈ 919). The dependent variable is the FHFA All-Transactions House Price Index (2024, CBSA-level). The primary regressor is the number of new residential units authorized by building permit (Census BPS, January 2026 snapshot). The baseline specification is ln(HPI<sub>i</sub>) = β<sub>0</sub> + β<sub>1</sub>·ln(Permits<sub>i</sub>) + γ′Controls<sub>i</sub> + ε<sub>i</sub>. We find [main result — complete after running regressions]. Results are [robust/not robust] to [alternative specification]. We discuss omitted variable bias from demand-side factors such as income and population, and multicollinearity between supply and demand indicators. Our findings [implication for housing policy].

---
## 0. Setup

In [ ]:
%pip install pandas numpy matplotlib statsmodels scipy xlrd openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

---
## 1. Introduction

**Research question:** Does greater housing supply — measured by new residential building permits — reduce home prices across U.S. metropolitan areas?

**Why it matters:** The U.S. housing affordability crisis has worsened steadily. By 2025, the cumulative supply gap reached an estimated 4.03 million homes (Realtor.com, 2026), with the median down payment requiring seven years of saving for a typical household. Over 42 million Americans spend more than 30% of income on housing. Standard supply-and-demand theory predicts that constrained supply — driven by zoning restrictions, permitting delays, and construction costs — pushes prices above competitive equilibrium. Yet the empirical magnitude of this relationship across metro areas remains a policy-relevant question.

**Approach:** We exploit cross-sectional variation in building permit activity across ~919 MSAs. Using OLS with heteroskedasticity-robust standard errors (HC3), we regress log home prices on log permits, controlling for demand-side factors. The log-log specification yields elasticity estimates directly interpretable as percentage responses.

---
## 2. Data and Summary Statistics

**Data sources:**
- **FHFA All-Transactions HPI (Annual, CBSA-level):** Federal Housing Finance Agency. Repeat-sales index using Fannie Mae/Freddie Mac loan data plus FHA and county recorder data. Covers all MSAs in the U.S. We use the 2024 annual index value (base year 1990).
- **Census Building Permits Survey (BPS), CBSA Monthly:** U.S. Census Bureau. Reports new privately-owned residential units authorized by building permit. We use the January 2026 monthly release as a cross-sectional snapshot of MSA-level supply activity. *Limitation: ideally we would use 12-month totals for 2024; the January 2026 snapshot introduces measurement noise but cross-sectional MSA rankings are persistent.*
- **ACS 2024 1-Year Estimates (to be added):** Median household income (B19013), total housing units (B25001), and population (B01003) at the MSA level. **Action required — re-download from data.census.gov selecting all MSAs, not United States. Save as** `data/acs_controls.csv`.

**Unit of observation:** MSA (Metropolitan Statistical Area), identified by 5-digit CBSA code.

**Variables:**
- *y* — `log_hpi`: Natural log of FHFA All-Transactions HPI 2024 (proxy for price level)
- *x* — `log_permits`: Natural log of total residential units permitted (Jan 2026)
- *Controls (pending ACS download)*: `log_income` (ln median HH income), `log_pop` (ln population)

### 2.1 Load and Prepare Data

In [2]:
hpi_raw = pd.read_excel('data/hpi_at_cbsa.xlsx', skiprows=5, header=None)
hpi_raw.columns = ['CBSA','Name','Year','Ann_Chg_Pct','HPI','HPI_1990base','HPI_2000base']
hpi_raw['CBSA'] = pd.to_numeric(hpi_raw['CBSA'], errors='coerce')
hpi_raw['Year'] = pd.to_numeric(hpi_raw['Year'], errors='coerce')
hpi_raw['HPI'] = pd.to_numeric(hpi_raw['HPI'], errors='coerce')
hpi_raw['Ann_Chg_Pct'] = pd.to_numeric(hpi_raw['Ann_Chg_Pct'], errors='coerce')

hpi = hpi_raw[(hpi_raw['CBSA'] >= 10000) & (hpi_raw['Year'] == 2024)][['CBSA','Name','HPI','Ann_Chg_Pct']].copy()
hpi['CBSA'] = hpi['CBSA'].astype(int)
hpi = hpi.dropna(subset=['HPI']).reset_index(drop=True)
print(f'FHFA HPI 2024: {len(hpi)} MSAs')
hpi.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/hpi_at_cbsa.xlsx'

In [ ]:
perm_raw = pd.read_excel('data/cbsamonthly_202601.xls', engine='xlrd', skiprows=6, header=None)
data_rows = perm_raw.iloc[2:].reset_index(drop=True)

perm = pd.DataFrame({
    'CBSA': pd.to_numeric(data_rows[1], errors='coerce'),
    'total_permits': pd.to_numeric(data_rows[11], errors='coerce')
})
perm = perm.dropna(subset=['CBSA','total_permits'])
perm['CBSA'] = perm['CBSA'].astype(int)
perm = perm.reset_index(drop=True)
print(f'BPS permits: {len(perm)} MSAs')
perm.head()

In [ ]:
# --- ACS CONTROLS (run once you re-download at MSA level from data.census.gov) ---
# acs_raw = pd.read_csv('data/acs_controls.csv', skiprows=[1])
# geo_col = [c for c in acs_raw.columns if 'geo' in c.lower()][0]
# acs_raw['CBSA'] = pd.to_numeric(acs_raw[geo_col].astype(str).str[-5:], errors='coerce')
# inc_col  = [c for c in acs_raw.columns if 'B19013_001' in c][0]
# pop_col  = [c for c in acs_raw.columns if 'B01003_001' in c][0]
# units_col = [c for c in acs_raw.columns if 'B25001_001' in c][0]
# acs = acs_raw[['CBSA', inc_col, pop_col, units_col]].copy()
# acs.columns = ['CBSA','median_income','population','housing_units']
# for col in ['median_income','population','housing_units']:
#     acs[col] = pd.to_numeric(acs[col], errors='coerce')
# acs = acs[acs['median_income'] > 0].dropna().reset_index(drop=True)
# print(f'ACS controls: {len(acs)} MSAs')
print('ACS controls not yet loaded — re-download at MSA level to enable full model')

In [ ]:
df = hpi.merge(perm, on='CBSA', how='inner')
print(f'Merged: {df.shape[0]} MSAs')
df.head()

In [ ]:
df['log_hpi']     = np.log(df['HPI'])
df['log_permits'] = np.log(df['total_permits'].replace(0, np.nan))
df = df.dropna(subset=['log_hpi','log_permits']).reset_index(drop=True)
print(f'Analysis sample (non-zero permits): {len(df)} MSAs')
print(f"Dropped (zero-permit MSAs): {hpi.merge(perm, on='CBSA').shape[0] - len(df)}")

### 2.2 Descriptive Statistics

In [ ]:
desc_vars = ['HPI','Ann_Chg_Pct','total_permits','log_hpi','log_permits']
desc = df[desc_vars].describe().T.round(3)
desc.columns = ['N','Mean','Std','Min','25%','50%','75%','Max']
print(desc.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df['HPI'], bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('FHFA HPI 2024 (Annual, NSA)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of HPI')

axes[1].hist(df['log_hpi'], bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
axes[1].set_xlabel('Log(HPI)')
axes[1].set_title('Distribution of Log(HPI)')

axes[2].hist(df['log_permits'], bins=40, color='coral', edgecolor='white', linewidth=0.4)
axes[2].set_xlabel('Log(Total Permits, Jan 2026)')
axes[2].set_title('Distribution of Log(Permits)')

plt.tight_layout()
plt.savefig('fig1_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

p1 = df[['total_permits','HPI']].dropna()
slope, intercept, r, pv, _ = stats.linregress(p1['total_permits'], p1['HPI'])
x = np.linspace(p1['total_permits'].min(), p1['total_permits'].max(), 200)
axes[0].scatter(p1['total_permits'], p1['HPI'], alpha=0.35, s=16,
               color='steelblue', edgecolors='k', linewidths=0.2)
axes[0].plot(x, slope*x+intercept, color='firebrick', lw=1.8)
axes[0].set_xlabel('Total Units Permitted (Jan 2026)')
axes[0].set_ylabel('FHFA HPI 2024 (NSA)')
axes[0].set_title('Permits vs. Home Prices (Levels)')
axes[0].text(0.04, 0.93, f'r = {r:.3f}, p = {pv:.4f}',
            transform=axes[0].transAxes, fontsize=9)

p2 = df[['log_permits','log_hpi']].dropna()
slope2, intercept2, r2, pv2, _ = stats.linregress(p2['log_permits'], p2['log_hpi'])
x2 = np.linspace(p2['log_permits'].min(), p2['log_permits'].max(), 200)
axes[1].scatter(p2['log_permits'], p2['log_hpi'], alpha=0.35, s=16,
               color='steelblue', edgecolors='k', linewidths=0.2)
axes[1].plot(x2, slope2*x2+intercept2, color='firebrick', lw=1.8)
axes[1].set_xlabel('Log(Total Units Permitted)')
axes[1].set_ylabel('Log(FHFA HPI 2024)')
axes[1].set_title('Log-Log Specification (Main)')
axes[1].text(0.04, 0.93, f'r = {r2:.3f}, p = {pv2:.4f}',
            transform=axes[1].transAxes, fontsize=9)

plt.tight_layout()
plt.savefig('fig2_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Bivariate correlation (log-log): r = {r2:.3f}')

---
## 3. Empirical Strategy

We estimate the following log-log OLS models:

**Model 1 — Bivariate (OVB benchmark):**

$$\ln(\text{HPI}_i) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \varepsilon_i$$

**Model 2 — Full specification (controls for demand-side OVB):**

$$\ln(\text{HPI}_i) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \beta_2 \ln(\text{Income}_i) + \beta_3 \ln(\text{Pop}_i) + \varepsilon_i$$

**Interpretation of β₁:** A 1% increase in permitted units is associated with a β₁% change in the HPI, holding other factors constant. Supply-demand theory predicts **β₁ < 0**: more supply → lower prices.

**Omitted Variable Bias (OVB):** The bivariate estimate of β₁ is likely upward biased (toward zero or positive) because demand-rich metros — with high income and population — simultaneously attract more construction *and* have higher prices. This positive correlation between permits and income/population biases β₁ upward in Model 1. Controlling for income and population in Model 2 should make β₁ more negative, moving it toward the true supply effect.

**Multicollinearity concern:** Log permits and log population will be positively correlated (large cities permit more units). We check Variance Inflation Factors (VIF) to ensure estimates are still interpretable.

**Estimation:** OLS with HC3 heteroskedasticity-robust standard errors throughout.

### 3.1 Estimation

In [ ]:
y = df['log_hpi']
X1 = sm.add_constant(df['log_permits'])
model1 = sm.OLS(y, X1).fit(cov_type='HC3')
print('=== Model 1: Bivariate ===')
print(model1.summary())

In [ ]:
# Model 2 requires ACS controls — placeholder until ACS re-downloaded
# Uncomment and run after adding acs_controls.csv

# df2 = df.merge(acs[['CBSA','median_income','population']], on='CBSA', how='inner')
# df2['log_income'] = np.log(df2['median_income'])
# df2['log_pop']    = np.log(df2['population'])
# df2 = df2.dropna(subset=['log_income','log_pop'])
# y2 = df2['log_hpi']
# X2 = sm.add_constant(df2[['log_permits','log_income','log_pop']])
# model2 = sm.OLS(y2, X2).fit(cov_type='HC3')
# print('=== Model 2: Full ===')
# print(model2.summary())

print('Model 2 pending ACS data at MSA level.')
print('Re-download B19013 + B01003 from data.census.gov selecting all MSAs.')

### 3.2 Diagnostic Awareness

In [ ]:
# OVB sanity check: compare Model 1 β₁ to what we expect after adding controls
# When Model 2 is available, run this comparison
print('Model 1 — log_permits coefficient:')
print(f"  β₁ = {model1.params['log_permits']:.4f}")
print(f"  p-value = {model1.pvalues['log_permits']:.4f}")
print(f"  R² = {model1.rsquared:.4f}")
print()
print('Expected direction of OVB: β₁ in Model 1 biased toward zero (or positive)')
print('→ Adding income/population controls (Model 2) should make β₁ more negative')

In [ ]:
# VIF check — only meaningful once Model 2 runs
# X_for_vif = df2[['log_permits','log_income','log_pop']].dropna()
# vif_data = pd.DataFrame({'Variable': X_for_vif.columns,
#     'VIF': [variance_inflation_factor(X_for_vif.values, i) for i in range(X_for_vif.shape[1])]})
# print(vif_data)
print('VIF available once Model 2 runs with ACS controls.')

---
## 4. Results

**Reading the table (Model 1):** The coefficient β₁ on log permits measures the elasticity of the FHFA HPI with respect to building permit activity. [Fill in after running: magnitude, sign, statistical significance.] The R² of [X]% suggests permits alone explain [X]% of the cross-MSA variation in log home prices.

**Note:** Model 2 (full specification) will be added once ACS controls are downloaded at MSA level. The key result to report is how β₁ changes between Model 1 and Model 2 — this is the OVB analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

fitted = model1.fittedvalues
resid = model1.resid

ax.scatter(fitted, resid, alpha=0.35, s=16, color='steelblue', edgecolors='k', linewidths=0.2)
ax.axhline(0, color='firebrick', lw=1.5, ls='--')
ax.set_xlabel('Fitted Values — Log(HPI)')
ax.set_ylabel('Residuals')
ax.set_title('Fig 3: Residuals vs. Fitted — Model 1')

plt.tight_layout()
plt.savefig('fig3_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sm.qqplot(model1.resid, line='s', ax=ax, alpha=0.45, markersize=4)
ax.set_title('Fig 4: Q-Q Plot of Residuals — Model 1')
plt.tight_layout()
plt.savefig('fig4_qq.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Robustness and Limitations

**Robustness check 1 — Annual HPI change as DV:** Instead of log price level, we use the annual percent change in HPI (2024) as the dependent variable. This tests whether permits predict *recent price momentum* rather than cumulative price levels.

**Robustness check 2 — Trimming outliers:** Dropping MSAs in the top and bottom 2.5% of permits to ensure results are not driven by very large or very small markets.

**Limitations:**
1. *Timing mismatch:* Permits measured in January 2026; HPI measured for full-year 2024. Annual 2023–2024 permit totals would be preferable.
2. *Reverse causality:* Higher prices may attract more construction (developers respond to profits). Our OLS estimate captures an association, not a clean causal effect. An IV using zoning strictness or geographic constraints would address this.
3. *Missing controls:* Without MSA-level income and population, Model 1 is subject to OVB. Model 2 (pending ACS download) will partially address this.
4. *Aggregation:* MSA-level analysis masks within-metro variation in zoning and neighborhood supply.

In [ ]:
rob1 = sm.OLS(df['Ann_Chg_Pct'], sm.add_constant(df['log_permits'])).fit(cov_type='HC3')
print('=== Robustness 1: DV = Annual HPI % Change (2024) ===')
print(f"β₁ = {rob1.params['log_permits']:.4f}  |  p = {rob1.pvalues['log_permits']:.4f}  |  R² = {rob1.rsquared:.4f}")

In [ ]:
lo, hi = df['log_permits'].quantile(0.025), df['log_permits'].quantile(0.975)
df_trim = df[(df['log_permits'] >= lo) & (df['log_permits'] <= hi)]
rob2 = sm.OLS(df_trim['log_hpi'], sm.add_constant(df_trim['log_permits'])).fit(cov_type='HC3')
print(f'=== Robustness 2: Trimmed sample (N={len(df_trim)}) ===')
print(f"β₁ = {rob2.params['log_permits']:.4f}  |  p = {rob2.pvalues['log_permits']:.4f}  |  R² = {rob2.rsquared:.4f}")
print(f"Compare to full sample β₁ = {model1.params['log_permits']:.4f}")

---
## 6. Conclusion

[Complete after running results.]

Using a cross-section of ~919 U.S. MSAs, we find that building permit activity is [positively/negatively] associated with home price levels. The log-log elasticity estimate suggests a 1% increase in permitted units corresponds to a [β₁]% change in the FHFA HPI. This result is [robust/sensitive] to trimming outlier markets. Once demand-side controls are added (Model 2), the coefficient [increases/decreases], consistent with upward OVB from high-demand metros simultaneously attracting supply and sustaining high prices. Our findings suggest that [policy implication — e.g., expanding permitting in supply-constrained metros could moderate price appreciation, but demand-side pressures limit the effect].

---
## References

- **FHFA House Price Index:** Federal Housing Finance Agency. *FHFA HPI All-Transactions CBSA Annual Data.* https://www.fhfa.gov/data/hpi/datasets (2025).
- **Census Building Permits Survey:** U.S. Census Bureau. *New Privately-Owned Residential Building Units Authorized, CBSA Monthly.* https://www.census.gov/construction/bps (2026).
- **ACS 2024:** U.S. Census Bureau. *American Community Survey 1-Year Estimates,* Tables B19013, B25001, B01003. https://data.census.gov (2024).
- Glaeser, E. L., Gyourko, J., & Saks, R. E. (2005). Why have housing prices gone up? *American Economic Review, 95*(2), 329–333.
- Realtor.com. (2026). *2026 Housing Supply Gap Report.*
- Wooldridge, J. M. (2019). *Introductory Econometrics: A Modern Approach* (7th ed.). Cengage.